# Data catalog audit

Questo notebook controlla la solidita del catalogo fonti: copertura per provider e tema, licenze, stato di riuso, link scoperti e priorita operative. Le celle producono tabelle filtrabili e grafici utili per scegliere le prossime normalizzazioni.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

for path in [Path.cwd(), *Path.cwd().parents]:
    helper_path = path / "notebooks" / "utils_notebooks.py"
    if helper_path.exists():
        sys.path.insert(0, str(helper_path.parent))
        break

from utils_notebooks import get_project_paths, plot_barh, plot_matrix, read_clean_csv, show_table

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

PATHS = get_project_paths()
ROOT = PATHS["root"]
CATALOG_DIR = PATHS["catalog"]
TABLES = PATHS["tables"]
METADATA = PATHS["metadata"]
PROCESSED = PATHS["processed"]
print("Project root:", ROOT)

## Dati letti

Carichiamo catalogo, moduli, ranking e link scoperti. Se una tabella manca, il notebook resta eseguibile e mostra esplicitamente il vuoto.

In [ ]:
catalog = read_clean_csv(CATALOG_DIR / "data_catalog.csv")
modules = read_clean_csv(CATALOG_DIR / "analysis_modules.csv")
discovered = read_clean_csv(CATALOG_DIR / "discovered_links.csv")
validated = read_clean_csv(TABLES / "validated_discovered_links.csv")
registry = read_clean_csv(TABLES / "dataset_registry.csv")
ranking = read_clean_csv(TABLES / "source_ranking.csv")

summary = pd.DataFrame([
    {"area": "fonti catalogate", "value": len(catalog)},
    {"area": "moduli analitici", "value": len(modules)},
    {"area": "righe discovery iniziale", "value": len(discovered)},
    {"area": "link validati", "value": len(validated)},
    {"area": "righe registry dataset", "value": len(registry)},
])
display(summary)
show_table(catalog, 10)

## Controllo schema

Le colonne minime tengono insieme pipeline, audit e notebook. La tabella indica presenza colonna e valori vuoti.

In [ ]:
required_columns = [
    "source_id", "provider", "dataset_name", "theme", "source_page_url", "access_type",
    "license", "redistribution_allowed", "output_subfolder", "enabled", "download_status",
]
rows = []
for column in required_columns:
    exists = column in catalog.columns
    missing_values = int(catalog[column].eq("").sum()) if exists and not catalog.empty else None
    rows.append({"column": column, "exists": exists, "empty_values": missing_values})
schema_check = pd.DataFrame(rows)
display(schema_check)

if not catalog.empty:
    duplicate_sources = catalog[catalog.duplicated("source_id", keep=False)].sort_values("source_id")
    print("Duplicate source_id:", len(duplicate_sources))
    show_table(duplicate_sources, 20)

## Copertura per provider, accesso e tema

Questi grafici aiutano a vedere se il catalogo e' bilanciato o concentrato su pochi provider/temi.

In [ ]:
if not catalog.empty:
    provider_counts = catalog.groupby("provider", dropna=False).size().reset_index(name="sources")
    access_counts = catalog.groupby("access_type", dropna=False).size().reset_index(name="sources")
    theme_counts = catalog.groupby("theme", dropna=False).size().reset_index(name="sources")
    plot_barh(provider_counts, "provider", "sources", "Fonti per provider", color="#0f766e")
    plot_barh(access_counts, "access_type", "sources", "Fonti per tipo di accesso", color="#7c3aed")
    plot_barh(theme_counts, "theme", "sources", "Fonti per tema", color="#b45309")
else:
    print("Catalogo non disponibile.")

In [ ]:
if not catalog.empty:
    matrix = catalog.pivot_table(index="provider", columns="theme", values="source_id", aggfunc="count", fill_value=0)
    display(matrix)
    plot_matrix(matrix, "Matrice provider x tema")

## Licenze, riuso e stato operativo

Fonti con licenza chiara, riuso consentito e accesso tabellare sono candidate migliori per download automatico.

In [ ]:
if not catalog.empty:
    license_counts = catalog.groupby("license", dropna=False).size().reset_index(name="sources")
    reuse_counts = catalog.groupby("redistribution_allowed", dropna=False).size().reset_index(name="sources")
    status_counts = catalog.groupby("download_status", dropna=False).size().reset_index(name="sources")
    plot_barh(license_counts, "license", "sources", "Fonti per licenza dichiarata", color="#334155")
    plot_barh(reuse_counts, "redistribution_allowed", "sources", "Fonti per riuso", color="#15803d")
    plot_barh(status_counts, "download_status", "sources", "Fonti per stato download", color="#be123c")

    unclear = catalog[(catalog["license"].str.contains("verificare", case=False, na=False)) | (catalog["redistribution_allowed"] != "yes")]
    columns = ["provider", "source_id", "dataset_name", "theme", "license", "redistribution_allowed", "access_type"]
    display(unclear[columns].sort_values(["provider", "theme"]))

## Link scoperti e validati

Confrontiamo discovery e validazione per distinguere fonti informative, report e veri candidati dati.

In [ ]:
if not discovered.empty:
    discovery_status = discovered.groupby(["provider", "status"], dropna=False).size().reset_index(name="links")
    show_table(discovery_status.sort_values("links", ascending=False), 30)
    plot_barh(discovery_status.groupby("provider")["links"].sum().reset_index(), "provider", "links", "Link o stati discovery per provider", color="#0369a1")
else:
    print("Discovery non disponibile.")

if not validated.empty:
    data_series = validated.get("is_data_file", pd.Series(False, index=validated.index)).fillna(False).astype(bool)
    report_series = validated.get("is_report_file", pd.Series(False, index=validated.index)).fillna(False).astype(bool)
    validated_summary = pd.DataFrame([
        {"metric": "link validati", "value": len(validated)},
        {"metric": "file dati", "value": int(data_series.sum())},
        {"metric": "report", "value": int(report_series.sum())},
        {"metric": "validazione senza errore", "value": int(validated.get("validation_error", pd.Series("", index=validated.index)).fillna("").eq("").sum())},
    ])
    display(validated_summary)
    ext_counts = validated.groupby("file_extension", dropna=False).size().reset_index(name="links")
    plot_barh(ext_counts, "file_extension", "links", "Link validati per formato", color="#9333ea")

## Ranking fonti

Il ranking e' la tabella da usare quando si decide dove investire tempo di normalizzazione.

In [ ]:
if not ranking.empty and "score" in ranking.columns:
    ranking["score_numeric"] = pd.to_numeric(ranking["score"], errors="coerce").fillna(0)
    top = ranking.sort_values("score_numeric", ascending=False).head(15)
    show_table(top[["provider", "source_id", "dataset_name", "theme", "access_type", "license", "score"]], 15)
    plot_barh(top, "source_id", "score_numeric", "Top fonti per punteggio operativo", xlabel="Score", color="#0f766e")
else:
    print("Ranking non disponibile.")

## Worklist catalogo

Tabella finale filtrabile: priorita, licenza, tema, accesso e presenza di candidati nel registry.

In [ ]:
if not catalog.empty:
    worklist = catalog.copy()
    if not ranking.empty and "score" in ranking.columns:
        worklist = worklist.merge(ranking[["source_id", "score"]], on="source_id", how="left", suffixes=("", "_ranking"))
    if not registry.empty and "source_id" in registry.columns:
        candidate_counts = registry.groupby("source_id", dropna=False).size().reset_index(name="candidate_links")
        data_counts = registry[registry.get("is_data_file", pd.Series(False, index=registry.index)).fillna(False).astype(bool)].groupby("source_id", dropna=False).size().reset_index(name="data_file_links")
        worklist = worklist.merge(candidate_counts, on="source_id", how="left").merge(data_counts, on="source_id", how="left")
    for column in ["candidate_links", "data_file_links", "score"]:
        if column in worklist.columns:
            worklist[column] = pd.to_numeric(worklist[column], errors="coerce").fillna(0)
    columns = ["provider", "source_id", "dataset_name", "theme", "access_type", "license", "redistribution_allowed", "download_status", "score", "candidate_links", "data_file_links", "source_page_url"]
    columns = [column for column in columns if column in worklist.columns]
    display(worklist[columns].sort_values(["score", "data_file_links", "candidate_links"], ascending=False).head(80))